In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 3D Channel Attention
class ChannelAttention3D(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super(ChannelAttention3D, self).__init__()
        reduced_channels = max(1, in_channels // reduction_ratio)
        self.global_avg_pool = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Conv3d(in_channels, reduced_channels, kernel_size=1, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv3d(reduced_channels, in_channels, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.global_avg_pool(x)
        avg_out = self.fc1(avg_out)
        avg_out = self.relu(avg_out)
        avg_out = self.fc2(avg_out)
        return self.sigmoid(avg_out) * x

# 3D Spatial Attention
class SpatialAttention3D(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention3D, self).__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv3d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        mean_out = torch.mean(x, dim=1, keepdim=True)
        combined = torch.cat([max_out, mean_out], dim=1)
        attention = self.conv(combined)
        return x * self.sigmoid(attention)

# Eidetic ConvLSTMCell with 3D convolution
class EideticConvLSTMCell3D(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3, padding=1):
        super(EideticConvLSTMCell3D, self).__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv3d(
            input_dim + hidden_dim, 7 * hidden_dim, kernel_size, padding=padding, bias=True
        )
        self.memory_gate = nn.Conv3d(hidden_dim, hidden_dim, kernel_size=1, padding=0, bias=True)

    def forward(self, x, h_prev, c_prev, m_prev):
        combined = torch.cat([x, h_prev], dim=1)
        conv_output = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g, cc_u, cc_r, cc_m = torch.split(conv_output, self.hidden_dim, dim=1)

        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_prev + i * g

        u = torch.sigmoid(cc_u)
        r = torch.sigmoid(cc_r)
        memory_update = torch.tanh(cc_m)
        m_next = u * m_prev + r * memory_update

        eidetic_output = torch.sigmoid(self.memory_gate(m_next))
        c_eidetic = eidetic_output * c_next

        h_next = o * torch.tanh(c_eidetic)
        return h_next, c_next, m_next

class EideticConvLSTM3D(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(EideticConvLSTM3D, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.layers = nn.ModuleList(
            [EideticConvLSTMCell3D(input_dim if i == 0 else hidden_dim, hidden_dim) for i in range(num_layers)]
        )

    def forward(self, x):
        b, _, d, h, w = x.size()
        h_states = [torch.zeros(b, self.hidden_dim, d, h, w, device=x.device) for _ in range(self.num_layers)]
        c_states = [torch.zeros(b, self.hidden_dim, d, h, w, device=x.device) for _ in range(self.num_layers)]
        m_states = [torch.zeros(b, self.hidden_dim, d, h, w, device=x.device) for _ in range(self.num_layers)]

        for i, layer in enumerate(self.layers):
            h_states[i], c_states[i], m_states[i] = layer(x, h_states[i], c_states[i], m_states[i])
            x = h_states[i]
        return x


class Encoder3D(nn.Module):
    def __init__(self, input_channels):
        super(Encoder3D, self).__init__()
        self.enc1 = nn.Sequential(
            nn.Conv3d(input_channels, input_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv3d(input_channels, input_channels, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.enc2 = nn.Sequential(
            nn.MaxPool3d(2),
            nn.Conv3d(input_channels, input_channels * 2, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.enc3 = nn.Sequential(
            nn.MaxPool3d(2),
            nn.Conv3d(input_channels * 2, input_channels * 4, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.enc4 = nn.Sequential(
            nn.MaxPool3d(2),
            nn.Conv3d(input_channels * 4, input_channels * 8, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.conv_lstm = EideticConvLSTM3D(input_channels * 8, input_channels * 8, num_layers=3)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)
        x4 = self.enc4(x3)
        x4 = self.conv_lstm(x4)
        return x1, x2, x3, x4

class Decoder3D(nn.Module):
    def __init__(self, output_channels):
        super(Decoder3D, self).__init__()
        self.up4 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),
            nn.Conv3d(output_channels * 8, output_channels * 4, kernel_size=3, padding=1),
        )
        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),
            nn.Conv3d(output_channels * 8, output_channels * 2, kernel_size=3, padding=1),
        )
        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False),
            nn.Conv3d(output_channels * 4, output_channels, kernel_size=3, padding=1),
        )

        self.lstm4 = EideticConvLSTM3D(output_channels * 4, output_channels * 4, num_layers=3)
        self.lstm3 = EideticConvLSTM3D(output_channels * 2, output_channels * 2, num_layers=3)
        self.lstm2 = EideticConvLSTM3D(output_channels, output_channels, num_layers=3)

        self.channel_attention1 = ChannelAttention3D(output_channels * 8)
        self.spatial_attention1 = SpatialAttention3D()
        self.channel_attention2 = ChannelAttention3D(output_channels * 4)
        self.spatial_attention2 = SpatialAttention3D()
        self.channel_attention3 = ChannelAttention3D(output_channels * 2)
        self.spatial_attention3 = SpatialAttention3D()

        self.final_conv = nn.Conv3d(output_channels * 2, 3, kernel_size=3, padding=1)

    def forward(self, x1, x2, x3, x4):
        x = self.up4(x4)
        x = self.lstm4(x)
        x = torch.cat([x, x3], dim=1)
        x = self.channel_attention1(x)
        x = self.spatial_attention1(x)

        x = self.up3(x)
        x = self.lstm3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.channel_attention2(x)
        x = self.spatial_attention2(x)

        x = self.up2(x)
        x = self.lstm2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.channel_attention3(x)
        x = self.spatial_attention3(x)

        return self.final_conv(x)

# 3D Autoencoder
class Autoencoder3D(nn.Module):
    def __init__(self, input_channels):
        super(Autoencoder3D, self).__init__()
        self.encoder = Encoder3D(input_channels)
        self.decoder = Decoder3D(input_channels)

    def forward(self, x):
        x1, x2, x3, x4 = self.encoder(x)
        return self.decoder(x1, x2, x3, x4)

# Test
if __name__ == "__main__":
    input_channels = 3
    model = Autoencoder3D(input_channels)
    sample_input = torch.randn(1, input_channels, 8, 256, 256)  # Batch size 1, 3 channels, 16 frames, 64x64
    output = model(sample_input)
    print("Input shape:", sample_input.shape)
    print("Output shape:", output.shape)


Input shape: torch.Size([1, 3, 8, 256, 256])
Output shape: torch.Size([1, 3, 8, 256, 256])


In [9]:
from torchsummary import summary
print(model)

Autoencoder3D(
  (encoder): Encoder3D(
    (enc1): Sequential(
      (0): Conv3d(3, 3, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): ReLU()
      (2): Conv3d(3, 3, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (3): ReLU()
    )
    (enc2): Sequential(
      (0): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): Conv3d(3, 6, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (2): ReLU()
    )
    (enc3): Sequential(
      (0): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): Conv3d(6, 12, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (2): ReLU()
    )
    (enc4): Sequential(
      (0): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): Conv3d(12, 24, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (2): ReLU()
    )
    (conv_lstm): EideticConvLSTM3D(
      (layers): ModuleList(
      

In [10]:
summary(model, input_size=(1, 3, 8, 256, 256), device='cuda' or 'cpu')

Layer (type:depth-idx)                             Output Shape              Param #
Autoencoder3D                                      --                        --
├─Encoder3D: 1                                     --                        --
│    └─EideticConvLSTM3D: 2                        --                        --
│    │    └─ModuleList: 3-1                        --                        655,488
├─Decoder3D: 1                                     --                        --
│    └─EideticConvLSTM3D: 2                        --                        --
│    │    └─ModuleList: 3-2                        --                        164,016
│    └─EideticConvLSTM3D: 2                        --                        --
│    │    └─ModuleList: 3-3                        --                        41,076
│    └─EideticConvLSTM3D: 2                        --                        --
│    │    └─ModuleList: 3-4                        --                        10,305
├─Encoder3D: 1-1 